# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides an end-to-end example for loading and exploring the [FAIR^2 dataset](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json) using the `mlcroissant` library.

### Dataset Source
The dataset is defined by a Croissant schema and is accessible via URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}")

## 2. Data Overview
Review available record sets and fields by their `@id`.

Let's get a list of record sets, and for each, inspect its fields. All entity references are by their `@id` per the Croissant schema specification.

In [ ]:
# List available record sets and their fields using their @id values

record_set_dict = {}
print("Available record sets and fields:")
for rs in dataset.metadata.record_sets:
    rs_id = rs.id
    rs_name = getattr(rs, 'name', '(no name)')
    record_set_dict[rs_id] = []
    print(f"- Record set @id: {rs_id}")
    print(f"  Name: {rs_name}")
    print(f"  Fields:")
    for field in rs.fields:
        field_id = field.id
        field_name = getattr(field, 'name', '(no name)')
        record_set_dict[rs_id].append(field_id)
        print(f"    • Field @id: {field_id}, name: {field_name}")
    print("")
if not record_set_dict:
    print("No record sets found in metadata.")

## 3. Data Extraction
Load data from all record sets into DataFrames for inspection and analysis. All selection is done using the `@id` values obtained above.

In [ ]:
# Extract data from all record sets into DataFrames
dataframes = {}
record_sets_list = list(record_set_dict.keys())
for record_set_id in record_sets_list:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded {len(df)} records for record set @id: {record_set_id}")
        else:
            print(f"No records found for record set @id: {record_set_id}")
    except Exception as ex:
        print(f"Error loading record set {record_set_id}: {ex}")

# Choose the first available dataframe for demo
if dataframes:
    selected_record_set_id = next(iter(dataframes))  # Take the first loaded
    selected_df = dataframes[selected_record_set_id]
    print(f"\nSample columns in DataFrame for record set @id: {selected_record_set_id}")
    print(selected_df.columns.tolist())
    display(selected_df.head())
else:
    print("No dataframes loaded. Check record sets and data access.")

## 4. Exploratory Data Analysis (EDA)
Apply basic processing steps: filter by a selected numeric field, normalize, and optionally group by a categorical field, referencing all columns by their `@id` values.

In [ ]:
# EDA Example: Select a numeric field and a group field by their @id
# Replace these @id values as appropriate for your dataset after reviewing above

from IPython.display import display
import numpy as np

# As an example, let's try to deduce a likely numeric field from the columns
if dataframes:
    df = selected_df
    numeric_candidate = None
    group_candidate = None
    # Guess numeric and grouping fields from column names/df.dtypes
    for col in df.columns:
        if np.issubdtype(df[col].dtype, np.number):
            numeric_candidate = col
            break
    # As group, pick the first non-numeric field
    for col in df.columns:
        if not np.issubdtype(df[col].dtype, np.number):
            group_candidate = col
            break

    if numeric_candidate is not None:
        numeric_field_id = numeric_candidate  # '@id' of a numeric field
        group_field_id = group_candidate      # '@id' of a group/category field, if available
        print(f"Using numeric field @id: {numeric_field_id}")
        if group_field_id:
            print(f"Using group field @id: {group_field_id}")

        threshold = df[numeric_field_id].mean()  # Use mean as example threshold

        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"\nFiltered records with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize the numeric field
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, norm_col]].head())

        # Group by the group field (if available)
        if group_field_id is not None and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame(name=f"mean_{numeric_field_id}")
            print(f"\nGrouped by {group_field_id} (mean of {numeric_field_id}):")
            display(grouped_df.head())
    else:
        print("No numeric fields detected for EDA. Check your schema and adjust field `@id`s accordingly.")
else:
    print("No dataframes to analyze.")

## 5. Visualization
Visualize distributions and relationships within the dataset. Below, we provide an example using a histogram and a boxplot for a selected numeric field, using the `@id` for column reference.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize only if numeric field was found in previous EDA cell
if dataframes and 'numeric_field_id' in locals() and numeric_field_id in df.columns:
    plt.figure(figsize=(12,4))
    plt.subplot(1,2,1)
    sns.histplot(df[numeric_field_id], bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")

    if group_field_id and group_field_id in df.columns:
        plt.subplot(1,2,2)
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print("No numeric or grouping field identified for visualization.")

## 6. Conclusion
In this notebook, we loaded and explored the FAIR^2 dataset using the `mlcroissant` library. We reviewed schema entities by their `@id`, extracted records into DataFrames by record set, performed basic data processing using dynamic variables, and visualized selected features.

To extend this workflow:
- Reference entity `@id`s directly from the overview in your analyses.
- Apply more advanced cleaning or modeling based on domain requirements.
- For robust reproducibility, always cite fields and columns using their stable `@id`.

For more examples and support, see the [mlcroissant documentation](https://mlcroissant.org/docs/intro/).